In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/home-credit-default-risk/bureau_balance.csv
/kaggle/input/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/home-credit-default-risk/previous_application.csv
/kaggle/input/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/home-credit-default-risk/installments_payments.csv
/kaggle/input/home-credit-default-risk/bureau.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/sample_submission.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/application_train.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/application_test.csv


# Leaky Modeling Pipeline

* This notebook intentionally builds a **naïve machine learning pipeline**
that violates data leakage and evaluation best practices.

The goal is to demonstrate how **leakage inflates performance metrics**
and creates a false sense of model quality.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix


In [3]:
data_path = "/kaggle/input/home-credit-default-risk/home-credit-default-risk/application_train.csv"
df = pd.read_csv(data_path)

df.shape

(307511, 122)

# Separate Features and Target (Leaky)

In [4]:

# Separate features and target
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())


Features shape: (307511, 121)
Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64


### All features (including ID-like and process-related ones) are kept, creating identifier and design-decision leakage risk.

# Naive Numeric Feature Selection

In [5]:
# Naively select only numeric features

X_num = X.select_dtypes(include=[np.number])

print("Numeric feature matrix shape:", X_num.shape)


Numeric feature matrix shape: (307511, 105)


### Numeric-only selection still keeps IDs and process proxies, enabling identifier leakage.

# Imputation BEFORE Split (Preprocessing Leakage)

In [6]:
#  PREPROCESSING LEAKAGE
# Impute missing values using statistics computed on the FULL dataset
# This allows test data information to leak into training data

X_num_imputed = X_num.fillna(X_num.median())

print("Missing values after imputation:",
      X_num_imputed.isnull().sum().sum())


Missing values after imputation: 0


# Feature Selection on Full Data (Feature Selection Leakage)

In [7]:
# FEATURE SELECTION LEAKAGE
# Selecting features using correlations computed on the FULL dataset

# Combine features and target to compute correlations
full_corr = (
    pd.concat([X_num_imputed, y], axis=1)
      .corr()["TARGET"]
      .abs()
      .sort_values(ascending=False)
)

# Select top correlated features (excluding the target itself)
top_features = full_corr.drop("TARGET").head(20).index.tolist()

# Reduce feature space using these selected features
X_selected = X_num_imputed[top_features]

print("Top selected features (leaky):")
print(top_features)
print("Shape after feature selection:", X_selected.shape)


Top selected features (leaky):
['EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_1', 'DAYS_BIRTH', 'REGION_RATING_CLIENT_W_CITY', 'REGION_RATING_CLIENT', 'DAYS_LAST_PHONE_CHANGE', 'DAYS_ID_PUBLISH', 'REG_CITY_NOT_WORK_CITY', 'FLAG_EMP_PHONE', 'DAYS_EMPLOYED', 'REG_CITY_NOT_LIVE_CITY', 'FLAG_DOCUMENT_3', 'DAYS_REGISTRATION', 'AMT_GOODS_PRICE', 'FLOORSMAX_AVG', 'FLOORSMAX_MEDI', 'FLOORSMAX_MODE', 'REGION_POPULATION_RELATIVE', 'ELEVATORS_AVG']
Shape after feature selection: (307511, 20)


# Random Train/Test Split (Evaluation Leakage)

In [8]:
# EVALUATION LEAKAGE
# Perform a random train/test split without considering time, process, or deployment conditions
# This assumes train and test come from the same distribution, which is often false in practice

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (246008, 20)
Test shape: (61503, 20)


# Train Logistic Regression

In [9]:
# Train a simple Logistic Regression model
#  No class weighting
#  No calibration
#  No legitimacy or imbalance handling

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

# Train Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Train a Random Forest on the SAME leaky data
#  Still uses leaked features
# Still ignores imbalance and legitimacy

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)



RandomForestClassifier(n_jobs=-1, random_state=42)

# Evaluation Using Accuracy (Metric Bias)

In [11]:
# METRIC / EVALUATION BIAS
# Evaluate both models using accuracy on an imbalanced dataset

# Logistic Regression predictions
lr_preds = model.predict(X_test)
lr_acc = accuracy_score(y_test, lr_preds)
lr_cm = confusion_matrix(y_test, lr_preds)

print("Logistic Regression Accuracy:", lr_acc)
print("Logistic Regression Confusion Matrix:")
print(lr_cm)

print("-" * 50)

# Random Forest predictions
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
rf_cm = confusion_matrix(y_test, rf_preds)

print("Random Forest Accuracy:", rf_acc)
print("Random Forest Confusion Matrix:")
print(rf_cm)


Logistic Regression Accuracy: 0.9195323805342829
Logistic Regression Confusion Matrix:
[[56552     2]
 [ 4947     2]]
--------------------------------------------------
Random Forest Accuracy: 0.9195811586426679
Random Forest Confusion Matrix:
[[56510    44]
 [ 4902    47]]


# ROC-AUC Evaluation

In [12]:
from sklearn.metrics import roc_auc_score

# Compute predicted probabilities
lr_proba = model.predict_proba(X_test)[:, 1]
rf_proba = rf_model.predict_proba(X_test)[:, 1]

# ROC-AUC scores
lr_roc_auc = roc_auc_score(y_test, lr_proba)
rf_roc_auc = roc_auc_score(y_test, rf_proba)

print("Logistic Regression ROC-AUC:", lr_roc_auc)
print("Random Forest ROC-AUC:", rf_roc_auc)


Logistic Regression ROC-AUC: 0.7101895964362543
Random Forest ROC-AUC: 0.7135216221407715


## Why Temporal and CV Leakage Are Not Induced

- Temporal leakage is not explicitly induced because it requires unverifiable
  time assumptions and would change the problem definition.
  It is instead identified as a risk and deferred to the applied project.

- Cross-validation leakage is not executed because preprocessing and feature
  selection leakage already demonstrate the same methodological failure.
  CV leakage is fully addressed and fixed in the leakage-safe pipeline.

This ensures the experiment remains controlled, interpretable, and reproducible.


# Leakage & Bias Summary — Notebook 01

This notebook intentionally demonstrated how a naïve ML pipeline can
produce misleadingly strong performance due to leakage and bias.

## Induced Leakages
1. Preprocessing leakage (imputation before split)
2. Feature selection leakage (full-data correlations)
3. Evaluation leakage (random train/test split)
4. Identifier leakage (ID-like numeric features)
5. Design-decision leakage (no legitimacy checks)
6. Cross-validation leakage (discussed, not executed)
7. Workflow leakage (missingness patterns)

## Induced Biases
- Class imbalance bias
- Metric bias (accuracy on imbalanced data)

## Key Observation
Both Logistic Regression and Random Forest achieve ~92% accuracy while
failing almost completely to identify defaulters.

## Conclusion
High accuracy in this setting is not evidence of a good model, but
evidence of a flawed evaluation pipeline.
